# Setup

In [ ]:
!nvidia-smi

In [ ]:
# # unzip data
# !unzip -q "/content/drive/MyDrive/Byte-sized-crop/S2Images.zip"

In [ ]:
!pip install iterative-stratification rasterio loguru -q

# Imports

In [ ]:
import os
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import cv2
from sklearn.model_selection import StratifiedKFold
import numpy as np
from tqdm import tqdm

# Data Processing

In [ ]:
from pathlib import Path
DATA_PATH = Path("/kaggle/input/byte-sized-agriculture-dataset/S2Images")
YEAR = 2024
MONTHS = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

mapper = {0: "Cocoa", 1: "Palm", 2: "Rubber"}
rev_mapper = {v: k for k, v in mapper.items()}

In [ ]:
# read data
train_df = pd.read_csv("/kaggle/input/byte-sized-agriculture-dataset/TrainDataset.csv")

In [ ]:
train_df["ID"] = train_df["ID"].apply(lambda x: "_".join(x.split("_")[:2]))

In [ ]:
def get_tiff_path(idx, month_idx, target):
    month_str = f"{month_idx:02d}"  # ensures 01, 02, ..., 12
    identifier = f"s2_{target}_{idx}_2024_{month_str}"

    tif_path = (
        DATA_PATH / ("train" if target != "Unknown" else "test") / f"{identifier}.tif"
    )
    if os.path.exists(tif_path):
        return tif_path
    return None  # Only return None if no match found

def add_tiff_path(df):
    data = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        tif_path = str(get_tiff_path(row.ID, MONTHS.index(row.month) + 1, row.Target))
        if tif_path:
            row["tifPath"] = tif_path
        data.append(row.to_dict())

    return pd.DataFrame(data)

In [ ]:
train_df = add_tiff_path(train_df)

In [ ]:
train_df = train_df[train_df.tifPath != "None"].reset_index(drop=True)

In [ ]:
train_df.head(3)

In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import DBSCAN

geo_train = gpd.read_file("/kaggle/input/byte-sized-agriculture-dataset/trainGeo.geojson")
geo_train["centroid_x"] = geo_train.geometry.apply(lambda geom: geom.centroid.x)
geo_train["centroid_y"] = geo_train.geometry.apply(lambda geom: geom.centroid.y)

coords = np.radians(geo_train[["centroid_y", "centroid_x"]].values)
kms_per_radian = 6371.0088
epsilon = 10 / kms_per_radian  # 10 km radius

db = DBSCAN(eps=epsilon, min_samples=5, algorithm="ball_tree", metric="haversine")
labels = db.fit_predict(coords)
geo_train["cluster"] = labels

n_split = 5
geo_train["folds"] = -1

# Combine cluster and Target to make a unique stratify label
geo_train["stratify_label"] = geo_train["cluster"].astype(str) + "_" + geo_train["Target"].astype(str)

skf = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=42)
for i, (_, vr) in enumerate(skf.split(geo_train, geo_train["stratify_label"])):
    geo_train.loc[geo_train.index[vr], "folds"] = i

# Now merge folds info into your train_df
train_df = pd.merge(train_df, geo_train[["ID", "folds"]], on=["ID"], how="left")

In [ ]:
train_df.head()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Predict on Test Set
test_df = pd.read_csv("/kaggle/input/byte-sized-agriculture-dataset/TestDataset.csv")
test_df["Target"] = "Unknown"
test_df["ID"] = test_df["ID"].apply(lambda x: "_".join(x.split("_")[:2]))
test_df = add_tiff_path(test_df)
test_df = test_df[test_df.tifPath != "None"].reset_index(drop=True)

# Modular

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np
import rasterio

def compute_indices(image):
    # image: shape (12, H, W) Sentinel-2 bands [B1, ..., B12]
    B = image
    eps = 1e-6
    indices = []
    # NDVI
    ndvi = (B[7] - B[3]) / (B[7] + B[3] + eps)
    indices.append(ndvi)
    # EVI
    evi = 2.5 * (B[7] - B[3]) / (B[7] + 6*B[3] - 7.5*B[1] + 1 + eps)
    indices.append(evi)
    # NDWI
    ndwi = (B[7] - B[10]) / (B[7] + B[10] + eps)
    indices.append(ndwi)
    # GNDVI
    gndvi = (B[7] - B[2]) / (B[7] + B[2] + eps)
    indices.append(gndvi)
    # NDMI
    ndmi = (B[7] - B[10]) / (B[7] + B[10] + eps)
    indices.append(ndmi)
    # MSI
    msi = B[10] / (B[7] + eps)
    indices.append(msi)
    # SAVI
    savi = 1.5 * (B[7] - B[3]) / (B[7] + B[3] + 0.5 + eps)
    indices.append(savi)
    # BNDVI
    bndvi = (B[7] - B[1]) / (B[7] + B[1] + eps)
    indices.append(bndvi)
    # NBR
    nbr = (B[7] - B[11]) / (B[7] + B[11] + eps)
    indices.append(nbr)
    # RECI
    reci = (B[7] / (B[4] + eps)) - 1
    indices.append(reci)
    indices = np.stack(indices, axis=0)  # (10, H, W)
    # Optionally, replace nan/inf with 0:
    indices = np.nan_to_num(indices, nan=0.0, posinf=0.0, neginf=0.0)
    return indices


# Normalization helper as discussed above
def normalize_band(
    band,
    method="minmax",
    band_index=None,
    sentinel2_stats=None,
    scale_max=10000
):
    band = np.ma.masked_invalid(band)
    if method == "minmax":
        band_min = band.min()
        band_max = band.max()
        if band_max > band_min:
            return (band - band_min) / (band_max - band_min)
        else:
            return band
    elif method == "zscore":
        assert band_index is not None and sentinel2_stats is not None
        mean, std = sentinel2_stats.get(band_index, (band.mean(), band.std()))
        if std > 0:
            return (band - mean) / std
        else:
            return band
    elif method == "fixed":
        return band / scale_max
    else:
        raise ValueError(f"Unknown normalization method: {method}")

# Your window reader
def process_window(tif_path, window=None):
    with rasterio.open(tif_path) as src:
        if window is not None:
            data = src.read(window=window)
        else:
            data = src.read()
        nodata_value = src.nodata
        if nodata_value is not None:
            data = np.ma.masked_equal(data, nodata_value)
    return data

class GroupedByteSizedDataset(Dataset):
    def __init__(self, dataframe, transform=None, to_train=True, normalization_method="minmax",
                 sentinel2_stats=None, scale_max=10000):
        self.transform = transform
        self.to_train = to_train
        self.normalization_method = normalization_method
        self.sentinel2_stats = sentinel2_stats
        self.scale_max = scale_max

        self.dataframe = dataframe
        # Group all rows by ID; store tifPath lists and first Target
        self.id_groups = dataframe.groupby('ID').agg({
            'tifPath': list,
            'Target': 'first'
        }).reset_index()
        self.ids = self.id_groups['ID'].tolist()
        self.unique_ids = self.ids     # <--- This line fixes Trainer compatibility!

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        group = self.id_groups.iloc[idx]
        tif_paths = group['tifPath']  # List of tif paths for this ID
        target = group['Target']
        images = []
        for tif_path in tif_paths:
            image = process_window(tif_path)  # (12, H, W)
            # Normalized RGB
            rgb_indices = [3, 2, 1, 8, 10, 11]
            bands = []
            for i, idx_band in enumerate(rgb_indices):
                band = image[idx_band]
                bands.append(
                    normalize_band(
                        band,
                        method=self.normalization_method,
                        band_index=idx_band+1,
                        sentinel2_stats=self.sentinel2_stats,
                        scale_max=self.scale_max,
                    )
                )
            img = np.stack(bands, axis=0)  # (3, H, W)
            # Compute and concatenate vegetation indices
            indices = compute_indices(image)  # (10, H, W)
            img = np.concatenate([img, indices], axis=0)  # (13, H, W)
            if isinstance(img, np.ma.MaskedArray):
                img = img.filled(0)
            img = np.moveaxis(img, 0, -1)  # (H, W, 13)  # keep for albumentations
            img = img.astype(np.float32)
            if self.transform:
                img = self.transform(image=img)["image"]
            img = torch.tensor(img, dtype=torch.float32)
            images.append(img)
        images = images[:7]
        images = torch.stack(images, dim=0)  # (N_imgs, C, H, W)
        if self.to_train:
            target = rev_mapper[target]
            return images, target
        else:
            return images


In [ ]:
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, emb_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Linear(emb_dim, num_heads)
        self.proj = nn.Linear(emb_dim * num_heads, emb_dim)  # Final projection

    def forward(self, x):
        # x: (batch, n_months, emb_dim)
        attn_scores = self.attn(x)  # (batch, n_months, num_heads)
        attn_weights = torch.softmax(attn_scores, dim=1)  # softmax over months
        weighted = (x.unsqueeze(-1) * attn_weights.unsqueeze(2)).sum(dim=1)  # (batch, emb_dim, num_heads)
        weighted = weighted.permute(0, 2, 1).contiguous().view(x.size(0), -1)  # (batch, emb_dim*num_heads)
        pooled = self.proj(weighted)  # (batch, emb_dim)
        return pooled

class AggregationModelAttnEnhanced(nn.Module):
    def __init__(self, base_name="swin_small_patch4_window7_224.ms_in22k", num_classes=3, n_months=7, attn_heads=4, dropout=0.3):
        super().__init__()
        
        num_input_channels = 3 + 3 + 10  # 3 RGB + 3 bands + 10 vegetation indices
        self.backbone = timm.create_model(base_name, pretrained=True, in_chans=num_input_channels, num_classes=0)
        test_input = torch.zeros(1,num_input_channels,224,224)
        out = self.backbone(test_input)
        emb_dim = out.shape[-1] if len(out.shape)==2 else out.shape[1]
        self.month_emb = nn.Embedding(n_months, emb_dim)
        self.attn_pool = MultiHeadAttentionPooling(emb_dim, num_heads=attn_heads)
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(emb_dim, num_classes)
        self.n_months = n_months

    def forward(self, x):
        b, n, c, h, w = x.shape
        x = x.view(-1, c, h, w)
        emb = self.backbone(x).view(b, n, -1)
        month_ids = torch.arange(n, device=x.device).unsqueeze(0).repeat(b,1)
        emb = emb + self.month_emb(month_ids)
        pooled = self.attn_pool(emb)
        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)
        out = self.classifier(pooled)
        return out


In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, input, target):
        logpt = F.log_softmax(input, dim=1)
        pt = torch.exp(logpt)
        logpt = logpt.gather(1, target.unsqueeze(1)).squeeze(1)
        pt = pt.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -(1 - pt) ** self.gamma * logpt
        if self.alpha is not None:
            loss = loss * self.alpha[target]
        if self.reduction == 'mean':
            return loss.mean()
        return loss

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
from loguru import logger
import pandas as pd
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import albumentations as A
from albumentations.pytorch import ToTensorV2

def mixup_data(x, y, alpha=0.4):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0.:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam
    
class Trainer:
    def __init__(self, model_class, dataset_class, train_df, test_df, mapper, folds,
                 fold_column="folds", num_classes=3, seed=42, img_size=64, batch_size=8, lr=1e-4, num_workers=2):
        self.model_class = model_class
        self.dataset_class = dataset_class
        self.train_df = train_df
        self.test_df = test_df
        self.mapper = mapper
        self.folds = folds
        self.fold_column = fold_column
        self.num_classes = num_classes
        self.seed = seed
        self.img_size = img_size
        self.batch_size = batch_size
        self.lr = lr
        self.num_workers = num_workers

    def set_seed(self, seed=None):
        if seed is None:
            seed = self.seed
        os.environ['PYTHONHASHSEED'] = str(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def seed_worker(self, worker_id):
        worker_seed = self.seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    def get_transforms(self):
        self.set_seed()
        train_transforms = A.Compose([
            A.OneOf(
                [A.RandomResizedCrop(size=(self.img_size, self.img_size), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=0.5),
                 A.Resize(self.img_size, self.img_size, p=0.5),],
                p=1.0
            ),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            # A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1, p=0.5),
            A.CoarseDropout(max_holes=8, max_height=self.img_size//8, max_width=self.img_size//8, p=0.4),
            ToTensorV2(),
        ])
        test_transforms = A.Compose([
            A.Resize(self.img_size, self.img_size),
            ToTensorV2(),
        ])
        return train_transforms, test_transforms

    def get_loaders(self, fold):
        train_transforms, test_transforms = self.get_transforms()
        train_data = self.train_df[self.train_df[self.fold_column] != fold].reset_index(drop=True)
        valid_data = self.train_df[self.train_df[self.fold_column] == fold].reset_index(drop=True)
        dataset_train = self.dataset_class(train_data, transform=train_transforms)
        dataset_valid = self.dataset_class(valid_data, transform=test_transforms)
        
        # Unique IDs for each group
        y_train = []
        for cur_id in dataset_train.unique_ids:
            rows = train_data[train_data["ID"] == cur_id]
            y_train.append(rows['class'].iloc[0] - 1)  # Assuming class starts from 1
        y_train = np.array(y_train)
        class_sample_count = np.array([(y_train == t).sum() for t in np.unique(y_train)])
        weight = 1. / class_sample_count
        samples_weight = np.array([weight[t] for t in y_train])
        samples_weight = torch.from_numpy(samples_weight).float()

        generator = torch.Generator()
        generator.manual_seed(self.seed + fold)
        sampler = WeightedRandomSampler(
            samples_weight,
            len(samples_weight),
            replacement=True,
            generator=generator
        )
        train_loader = DataLoader(
            dataset_train,
            batch_size=self.batch_size,
            sampler=sampler,
            num_workers=self.num_workers,
            shuffle=False,
            worker_init_fn=self.seed_worker,
            generator=generator,
            drop_last=False,
        )
        valid_loader = DataLoader(
            dataset_valid,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            worker_init_fn=self.seed_worker,
            drop_last=False,
        )
        return train_loader, valid_loader, valid_data
    
    def train_fold(self, fold, num_epochs=18, model_save_dir="models", label_smoothing=0.1, patience=7):
        self.set_seed(self.seed + fold)
        os.makedirs(model_save_dir, exist_ok=True) 
        model_save_path = os.path.join(model_save_dir, f"best_model_fold{fold}.pth")

        train_loader, valid_loader, valid_data = self.get_loaders(fold)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = self.model_class().to(device)

        y_train = []
        for cur_id in train_loader.dataset.unique_ids:
            rows = train_loader.dataset.dataframe[train_loader.dataset.dataframe["ID"] == cur_id]
            y_train.append(rows['class'].iloc[0] - 1)
        y_train = np.array(y_train)
        class_weights = compute_class_weight('balanced', classes=np.arange(self.num_classes), y=y_train)
        class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
        alpha = torch.tensor([1.2, 1.5, 1.5], device=device)  # Cocoa, Palm, Rubber
        criterion = FocalLoss(alpha=alpha, gamma=2)

        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, factor=0.5, verbose=True)

        best_f1 = 0.0
        patience_counter = 0

        logger.info(f"Starting training for fold {fold}")
        for epoch in range(num_epochs):
            self.set_seed(self.seed + fold * 100 + epoch)
            model.train()
            epoch_loss = 0.0
            correct, total = 0, 0
            all_preds, all_targets = [], []

            for images, targets in tqdm(train_loader, desc=f"Fold {fold} | Epoch {epoch+1}/{num_epochs} - Training"):
                images, targets = images.to(device), targets.to(device)
                
                use_mixup = np.random.rand() < 0.2   # 50% probability
                if use_mixup:
                    images, targets_a, targets_b, lam = mixup_data(images, targets, alpha=0.4)
                    outputs = model(images)
                    loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, targets)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
                _, preds = torch.max(outputs, 1)
                if use_mixup:
                    # For accuracy/F1, we can use targets_a for a rough estimate.
                    correct += (preds == targets_a).sum().item()
                    total += targets_a.size(0)
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(targets_a.cpu().numpy())
                else:
                    correct += (preds == targets).sum().item()
                    total += targets.size(0)
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(targets.cpu().numpy())

            train_loss = epoch_loss / len(train_loader)
            train_acc = correct / total
            train_f1 = f1_score(all_targets, all_preds, average='macro')

            val_loss, val_acc, val_f1 = self.validate(model, valid_loader, class_weights)
            scheduler.step(val_f1)
            logger.info(f"Fold {fold} | Epoch {epoch+1}/{num_epochs} | "
                        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f} | "
                        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), model_save_path)
                logger.info(f"Fold {fold} | >> Saved best model with val F1: {best_f1:.4f}")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    logger.info(f"Early stopping at epoch {epoch+1} (no improvement in {patience} epochs)")
                    break

        logger.info(f"Finished training for fold {fold}, best val F1: {best_f1:.4f}")
        return model_save_path

    def validate(self, model, valid_loader, class_weights):
        device = next(model.parameters()).device
        model.eval()
        val_loss = 0.0
        val_correct, val_total = 0, 0
        val_all_preds, val_all_targets = [], []
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
        with torch.no_grad():
            for images, targets in valid_loader:
                images, targets = images.to(device), targets.to(device)
                outputs = model(images)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == targets).sum().item()
                val_total += targets.size(0)
                val_all_preds.extend(preds.cpu().numpy())
                val_all_targets.extend(targets.cpu().numpy())
        val_loss /= len(valid_loader)
        val_acc = val_correct / val_total
        val_f1 = f1_score(val_all_targets, val_all_preds, average='macro')
        return val_loss, val_acc, val_f1

    @staticmethod
    def apply_tta(model, images):
        # images: (batch, num_crops, C, H, W)
        probas_list = []
        probas_list.append(F.softmax(model(images), dim=1))  # original
        # probas_list.append(F.softmax(model(torch.flip(images, dims=[-1])), dim=1))  # horizontal flip
        mean_probas = torch.stack(probas_list, dim=0).mean(dim=0)
        return mean_probas

    def predict_with_tta(self, model, loader, targets_provided=True):
        device = next(model.parameters()).device
        preds, true_vals, pred_probas = [], [], []
        model.eval()
        with torch.no_grad():
            for batch in tqdm(loader, desc="TTA Inference"):
                if targets_provided:
                    images, targets = batch
                    true_vals.append(targets.cpu().numpy())
                else:
                    images = batch
                images = images.to(device)
                tta_probas = self.apply_tta(model, images)
                probas = tta_probas.cpu().numpy()
                pred_classes = tta_probas.argmax(dim=1).cpu().numpy()
                preds.append(pred_classes)
                pred_probas.append(probas)
        preds = np.concatenate(preds, axis=0)
        pred_probas = np.concatenate(pred_probas, axis=0)
        if targets_provided:
            true_vals = np.concatenate(true_vals, axis=0)
            return preds, pred_probas, true_vals
        else:
            return preds, pred_probas

    # Other methods (cross_val_training, cross_val_inference, evaluate_grouped, test_infer_cv_ensemble) 
    # stay the same! (Just ensure all loaders and models are updated as above.)
    def cross_val_training(self, num_epochs=18, model_save_dir="models", label_smoothing=0.1, patience=7):
        all_model_paths = []
        for fold in self.folds:
            model_path = self.train_fold(
                fold=fold,
                num_epochs=num_epochs,
                model_save_dir=model_save_dir,
                label_smoothing=label_smoothing,
                patience=patience
            )
            all_model_paths.append(model_path)
        logger.info(f"All folds trained: {all_model_paths}")
        return all_model_paths

    def cross_val_inference(self, model_save_dir="models", oof_dir="train_oof_proba"):
        os.makedirs(oof_dir, exist_ok=True)
        valid_data_all = []
        for fold in self.folds:
            model = self.model_class()
            model_path = os.path.join(model_save_dir, f"best_model_fold{fold}.pth")
            model.load_state_dict(torch.load(model_path, map_location="cpu"))
            model = model.cuda() if torch.cuda.is_available() else model
    
            _, valid_loader, valid_data = self.get_loaders(fold)
            preds, pred_probas, true_vals = self.predict_with_tta(model, valid_loader)
    
            # We have one prediction per group (unique ID) already.
            valid_data_grouped = pd.DataFrame({
                'ID': valid_loader.dataset.unique_ids,
                'gt': true_vals,
                'pred_proba': list(pred_probas)
            })
    
            valid_data_all.append(valid_data_grouped)
            fold_oof_path = os.path.join(oof_dir, f"fold{fold}.csv")
            save_df = valid_data_grouped.copy()
            save_df['pred_proba'] = save_df['pred_proba'].apply(lambda x: ','.join([f'{v:.8f}' for v in x]))
            save_df.to_csv(fold_oof_path, index=False)
            logger.info(f"Fold {fold} OOF probabilities saved to {fold_oof_path}")
    
            acc = accuracy_score(valid_data_grouped['gt'], pred_probas.argmax(axis=1))
            f1 = f1_score(valid_data_grouped['gt'], pred_probas.argmax(axis=1), average='macro')
            print(f"Fold {fold} | Macro F1: {f1:.4f}")
    
        valid_data_all = pd.concat(valid_data_all).reset_index(drop=True)
        logger.info("Validation inference complete.")
        return valid_data_all

    def test_infer_cv_ensemble(self, model_save_dir="models", submission_path="submission.csv", 
                           id_col="ID", oof_dir="test_oof_proba", fold_indices=None):
        os.makedirs(oof_dir, exist_ok=True)
        test_df = self.test_df.copy()
        _, test_transforms = self.get_transforms()
        dataset_test = self.dataset_class(test_df, transform=test_transforms, to_train=False)
        test_loader = DataLoader(dataset_test, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, worker_init_fn=self.seed_worker)
        all_fold_pred_probas = []
    
        for fold in self.folds:
            model = self.model_class()
            model_path = os.path.join(model_save_dir, f"best_model_fold{fold}.pth")
            model.load_state_dict(torch.load(model_path, map_location="cpu"))
            model = model.cuda() if torch.cuda.is_available() else model
            _, pred_probas = self.predict_with_tta(model, test_loader, targets_provided=False)
            all_fold_pred_probas.append(pred_probas)
            fold_oof_path = os.path.join(oof_dir, f"fold{fold}.csv")
            fold_df = pd.DataFrame({
                id_col: dataset_test.unique_ids,
                'pred_proba': [','.join([f'{v:.8f}' for v in x]) for x in pred_probas]
            })
            fold_df.to_csv(fold_oof_path, index=False)
            logger.info(f"Test probabilities for fold {fold} saved to {fold_oof_path}")
    
        logger.info("All folds' OOF saved, now blending...")
    
        # Blend (average) over folds
        all_fold_pred_probas = np.stack(all_fold_pred_probas)  # (num_folds, num_samples, num_classes)
        mean_pred_probas = all_fold_pred_probas.mean(axis=0)   # (num_samples, num_classes)
    
        # Prepare submission
        predicted_targets = mean_pred_probas.argmax(axis=1)
        # Map back to original target labels
        submission_df = pd.DataFrame({
            id_col: dataset_test.unique_ids,
            'Target': [self.mapper[x] for x in predicted_targets]
        })
        submission_df.to_csv(submission_path, index=False)
        logger.info(f"Final blended submission saved to {submission_path}")
        return submission_df

In [ ]:
from loguru import logger

# Define your Trainer:
trainer = Trainer(
    model_class=AggregationModelAttnEnhanced,           # <-- your new model!
    dataset_class=GroupedByteSizedDataset,  # <-- your new grouped dataset!
    train_df=train_df,
    test_df=test_df,
    mapper=mapper,    # e.g. {0: "wheat", 1: "rice", 2: "corn"}
    folds=[0, 1, 2 , 3, 4], # 0, 1, 2 , 3, 4
    fold_column="folds",
    num_classes=3,
    seed=42,
    img_size=224,
    batch_size=8,     # Reduce batch size if you get OOM with groups!
    lr=1e-4,
    num_workers=os.cpu_count()
)

# Train all folds
trainer.cross_val_training(num_epochs=65, model_save_dir="models", label_smoothing=0.0, patience=16)

# Inference and evaluation on validation sets across all folds
valid_data_all = trainer.cross_val_inference(model_save_dir="models")
# Directly compute F1 and accuracy from valid_data_all:
from sklearn.metrics import f1_score, accuracy_score
y_true = valid_data_all['gt']
y_pred = np.array([np.argmax([float(x) for x in row.split(',')]) if isinstance(row, str) else np.argmax(row) for row in valid_data_all['pred_proba']])
print(f"Validation macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Validation accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Test ensemble prediction and submission
trainer.test_infer_cv_ensemble(model_save_dir="models", submission_path="submission.csv")


In [ ]:
print(pd.read_csv('/kaggle/working/submission.csv')["Target"].value_counts())
print(30*"--")
print(pd.read_csv('/kaggle/working/submission.csv')["Target"].value_counts(normalize=True))

In [ ]:
# Q ! is there a relation between missed months and target ?